In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load and inspect the dataset

If your CSV file is stored somewhere else, update the path in `pd.read_csv()`.

In [ ]:
df = pd.read_csv("insurance.csv")

display(df.head())
print("Dataset shape:", df.shape)
print("\\nColumn information:")
df.info()

### Check data quality

Inspect missing values, duplicate rows, and descriptive statistics. Decide whether any cleaning is needed before continuing.

In [ ]:
print("Missing values:")
display(df.isna().sum())

print("Number of duplicated rows:", df.duplicated().sum())

print("Descriptive statistics:")
display(df.describe(include="all").T)

## 3. Exploratory Data Analysis (EDA)

Use visualizations to explore the relationships between the features and insurance charges.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="charges", kde=True)
plt.title("Distribution of Medical Insurance Charges")
plt.xlabel("Insurance charges")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=0.7)
plt.title("Age vs. Insurance Charges")
plt.show()


plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="bmi", y="charges", hue="smoker", alpha=0.7)
plt.title("BMI vs. Insurance Charges")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="smoker", y="charges")
plt.title("Insurance Charges by Smoking Status")
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="region", y="charges")
plt.title("Insurance Charges by Region")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df.select_dtypes(include=np.number).corr(),
            annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Numerical Features")
plt.show()

## 4. Separate features and target

The target is `charges`. All other columns are input features.

We split the data **before fitting any preprocessing transformations**. This prevents information from the test set leaking into the training process.

In [ ]:
X = df.drop(columns="charges")
y = df["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

## 5. Traditional preprocessing (manual approach)

We have numerical and categorical features, so we need different transformations:

- Numerical columns: fill missing values with the training median, then standardize.
- Categorical columns: fill missing values with the most frequent training value, then apply one-hot encoding.

This dataset normally has no missing values, but the imputation steps make the code more reusable.

**Important:** We fit the imputer, scaler, and encoder on training data only.

In [ ]:
numeric_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

X_train_manual = X_train.copy()
X_test_manual = X_test.copy()


numeric_medians = X_train_manual[numeric_features].median()

X_train_num = X_train_manual[numeric_features].fillna(numeric_medians)
X_test_num = X_test_manual[numeric_features].fillna(numeric_medians)

scaler = StandardScaler()

X_train_num_scaled = scaler.fit_transform(X_train_num)
X_test_num_scaled = scaler.transform(X_test_num)



categorical_modes = X_train_manual[categorical_features].mode().iloc[0]

X_train_cat = X_train_manual[categorical_features].fillna(categorical_modes)
X_test_cat = X_test_manual[categorical_features].fillna(categorical_modes)


encoder = OneHotEncoder(handle_unknown="ignore")

X_train_cat_encoded = encoder.fit_transform(X_train_cat)
X_test_cat_encoded = encoder.transform(X_test_cat)

encoded_feature_names = encoder.get_feature_names_out(categorical_features)

X_train_cat_encoded = pd.DataFrame(
    X_train_cat_encoded,
    columns=encoded_feature_names,
    index=X_train.index
)

X_test_cat_encoded = pd.DataFrame(
    X_test_cat_encoded,
    columns=encoded_feature_names,
    index=X_test.index
)

# --- Combine numerical and categorical features ---
X_train_scaled = pd.concat(
    [
        pd.DataFrame(
            X_train_num_scaled,
            columns=numeric_features,
            index=X_train.index
        ),
        X_train_cat_encoded
    ],
    axis=1
)

X_test_scaled = pd.concat(
    [
        pd.DataFrame(
            X_test_num_scaled,
            columns=numeric_features,
            index=X_test.index
        ),
        X_test_cat_encoded
    ],
    axis=1
)

print("Prepared training data:", X_train_scaled.shape)
print("Prepared testing data:", X_test_scaled.shape)
display(X_train_scaled.head())

### Prepare features for tree-based models

Decision Trees and Random Forests do not require standardization. We will use the numerical features in their original scale, together with the one-hot encoded categorical features.

In [ ]:
X_train_tree = pd.concat(
    [
        X_train_num.reset_index(drop=True),
        X_train_cat_encoded.reset_index(drop=True)
    ],
    axis=1
)

X_test_tree = pd.concat(
    [
        X_test_num.reset_index(drop=True),
        X_test_cat_encoded.reset_index(drop=True)
    ],
    axis=1
)


y_train_tree = y_train.reset_index(drop=True)
y_test_tree = y_test.reset_index(drop=True)

print("Tree training data:", X_train_tree.shape)
print("Tree testing data:", X_test_tree.shape)

## 6. Evaluation helper

We will use the same evaluation function for each model.

- **MAE:** average absolute prediction error. Lower is better.
- **MSE:** average squared prediction error. Lower is better.
- **RMSE:** square root of MSE, expressed in the target's units. Lower is better.
- **R²:** proportion of target variance explained by the model. Higher is generally better.

An R² score can be negative when the model performs worse than predicting the target mean.

In [ ]:
def evaluate_regression(model_name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    return {
        "Model": model_name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }


results = []

## 7. Model 1 – Linear Regression

Linear Regression is our baseline model. It attempts to model the relationship between the input features and the continuous target.

**Import:** `from sklearn.linear_model import LinearRegression`

We use the scaled numerical features and one-hot encoded categorical features.

In [ ]:
linear_model = LinearRegression()
linear_model.fit(X_train_scaled, y_train)

linear_train_pred = linear_model.predict(X_train_scaled)
linear_test_pred = linear_model.predict(X_test_scaled)

results.append(
    evaluate_regression("Linear Regression", y_test, linear_test_pred)
)

print("Training R2:", r2_score(y_train, linear_train_pred))
print("Testing R2:", r2_score(y_test, linear_test_pred))

coefficients = pd.DataFrame({
    "Feature": X_train_scaled.columns,
    "Coefficient": linear_model.coef_
})

display(coefficients)
print("Intercept:", linear_model.intercept_)

## 8. Model 2 – Polynomial Regression

Polynomial Regression creates additional features from the original numerical features, then fits a Linear Regression model.

We will apply the polynomial transformation to the numerical features only. The one-hot encoded categorical features are added afterward.

**Imports:**
- `from sklearn.preprocessing import PolynomialFeatures`
- `from sklearn.linear_model import LinearRegression`

Try degree 2 first. Then experiment with degree 3.

In [ ]:
poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_num_poly = poly.fit_transform(X_train_num_scaled)
X_test_num_poly = poly.transform(X_test_num_scaled)

X_train_poly = np.concatenate(
    [X_train_num_poly, X_train_cat_encoded.to_numpy()],
    axis=1
)

X_test_poly = np.concatenate(
    [X_test_num_poly, X_test_cat_encoded.to_numpy()],
    axis=1
)

polynomial_model = LinearRegression()
polynomial_model.fit(X_train_poly, y_train)

poly_train_pred = polynomial_model.predict(X_train_poly)
poly_test_pred = polynomial_model.predict(X_test_poly)

results.append(
    evaluate_regression("Polynomial Regression (degree 2)", y_test, poly_test_pred)
)

print("Training R2:", r2_score(y_train, poly_train_pred))
print("Testing R2:", r2_score(y_test, poly_test_pred))
print("Number of numerical features before:", X_train_num_scaled.shape[1])
print("Number of polynomial numerical features:", X_train_num_poly.shape[1])

### Experiment: polynomial degree

Compare degrees 2 and 3. The following cell repeats the transformation correctly for each degree and records the test metrics.

Be aware that repeatedly selecting models based on test-set performance can make the test score optimistic. In a larger project, use a separate validation set or cross-validation for model selection, and keep the test set for final evaluation.

In [ ]:
polynomial_experiment_results = []

for degree in [2, 3]:
    poly_experiment = PolynomialFeatures(
        degree=degree,
        include_bias=False
    )

    train_num_poly = poly_experiment.fit_transform(X_train_num_scaled)
    test_num_poly = poly_experiment.transform(X_test_num_scaled)

    train_features = np.concatenate(
        [train_num_poly, X_train_cat_encoded.to_numpy()],
        axis=1
    )
    test_features = np.concatenate(
        [test_num_poly, X_test_cat_encoded.to_numpy()],
        axis=1
    )

    model = LinearRegression()
    model.fit(train_features, y_train)
    predictions = model.predict(test_features)

    polynomial_experiment_results.append(
        evaluate_regression(f"Polynomial degree {degree}", y_test, predictions)
    )

display(pd.DataFrame(polynomial_experiment_results))

## 9. Model 3 – Ridge Regression (L2 Regularization)

Ridge adds an L2 penalty that discourages large coefficients.

**Import:** `from sklearn.linear_model import Ridge`

Important hyperparameter:
- `alpha`: regularization strength. Larger values mean a stronger penalty.

Feature scaling is important for Ridge because the penalty is affected by coefficient size.

In [ ]:
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)

ridge_train_pred = ridge_model.predict(X_train_scaled)
ridge_test_pred = ridge_model.predict(X_test_scaled)

results.append(
    evaluate_regression("Ridge (alpha=1.0)", y_test, ridge_test_pred)
)

print("Training R2:", r2_score(y_train, ridge_train_pred))
print("Testing R2:", r2_score(y_test, ridge_test_pred))

### Experiment: Ridge alpha

Try different regularization strengths and compare the results.

In [ ]:
ridge_experiment_results = []

for alpha in [0.01, 0.1, 1, 10, 100]:
    model = Ridge(alpha=alpha)
    model.fit(X_train_scaled, y_train)

    train_predictions = model.predict(X_train_scaled)
    test_predictions = model.predict(X_test_scaled)

    ridge_experiment_results.append({
        "Alpha": alpha,
        "Train R2": r2_score(y_train, train_predictions),
        **evaluate_regression(f"Ridge alpha={alpha}", y_test, test_predictions)
    })

display(pd.DataFrame(ridge_experiment_results))

## 10. Model 4 – Lasso Regression (L1 Regularization)

Lasso adds an L1 penalty. It can shrink some coefficients to exactly zero, which can make it useful for feature selection.

**Import:** `from sklearn.linear_model import Lasso`

Important hyperparameters:
- `alpha`: regularization strength.
- `max_iter`: maximum number of optimization iterations.

Feature scaling is important for Lasso too.

In [ ]:
lasso_model = Lasso(alpha=0.1, max_iter=10000)
lasso_model.fit(X_train_scaled, y_train)

lasso_train_pred = lasso_model.predict(X_train_scaled)
lasso_test_pred = lasso_model.predict(X_test_scaled)

results.append(
    evaluate_regression("Lasso (alpha=0.1)", y_test, lasso_test_pred)
)

print("Training R2:", r2_score(y_train, lasso_train_pred))
print("Testing R2:", r2_score(y_test, lasso_test_pred))

lasso_coefficients = pd.DataFrame({
    "Feature": X_train_scaled.columns,
    "Coefficient": lasso_model.coef_
})

display(lasso_coefficients.sort_values("Coefficient", key=abs, ascending=False))
print("Number of zero coefficients:", np.sum(lasso_model.coef_ == 0))

### Experiment: Lasso alpha

Compare the following values. Observe how the number of zero coefficients changes as regularization becomes stronger.

In [ ]:
lasso_experiment_results = []

for alpha in [0.01, 0.1, 1, 10]:
    model = Lasso(alpha=alpha, max_iter=20000)
    model.fit(X_train_scaled, y_train)

    train_predictions = model.predict(X_train_scaled)
    test_predictions = model.predict(X_test_scaled)

    lasso_experiment_results.append({
        "Alpha": alpha,
        "Zero coefficients": int(np.sum(model.coef_ == 0)),
        "Train R2": r2_score(y_train, train_predictions),
        **evaluate_regression(f"Lasso alpha={alpha}", y_test, test_predictions)
    })

display(pd.DataFrame(lasso_experiment_results))

## 11. Model 5 – Decision Tree Regressor

A Decision Tree learns a sequence of decision rules and can capture nonlinear relationships.

**Import:** `from sklearn.tree import DecisionTreeRegressor`

Important hyperparameters:
- `max_depth`: maximum depth of the tree.
- `min_samples_split`: minimum samples required to split an internal node.
- `min_samples_leaf`: minimum samples allowed in a leaf.
- `random_state`: makes results reproducible.

Decision Trees do not require feature scaling.

In [ ]:
tree_model = DecisionTreeRegressor(
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=RANDOM_STATE
)

tree_model.fit(X_train_tree, y_train_tree)

tree_train_pred = tree_model.predict(X_train_tree)
tree_test_pred = tree_model.predict(X_test_tree)

results.append(
    evaluate_regression("Decision Tree", y_test_tree, tree_test_pred)
)

print("Training R2:", r2_score(y_train_tree, tree_train_pred))
print("Testing R2:", r2_score(y_test_tree, tree_test_pred))

### Experiment: Decision Tree hyperparameters

Try different tree depths and minimum leaf sizes. Compare training and testing scores.

In [ ]:
tree_experiment_results = []

for depth in [3, 5, 10, None]:
    for leaf_size in [1, 5, 10]:
        model = DecisionTreeRegressor(
            max_depth=depth,
            min_samples_leaf=leaf_size,
            min_samples_split=2,
            random_state=RANDOM_STATE
        )

        model.fit(X_train_tree, y_train_tree)

        train_predictions = model.predict(X_train_tree)
        test_predictions = model.predict(X_test_tree)

        tree_experiment_results.append({
            "max_depth": depth,
            "min_samples_leaf": leaf_size,
            "Train R2": r2_score(y_train_tree, train_predictions),
            **evaluate_regression(
                f"Tree depth={depth}, leaf={leaf_size}",
                y_test_tree,
                test_predictions
            )
        })

tree_experiment_df = pd.DataFrame(tree_experiment_results)
display(tree_experiment_df.sort_values("RMSE"))

## 12. Model 6 – Random Forest Regressor

Random Forest is an ensemble algorithm. It trains multiple Decision Trees and combines their predictions.

**Import:** `from sklearn.ensemble import RandomForestRegressor`

Important hyperparameters:
- `n_estimators`: number of trees.
- `max_depth`: maximum depth of each tree.
- `min_samples_split`: minimum samples required to split a node.
- `min_samples_leaf`: minimum samples required in a leaf.
- `max_features`: number of features considered when searching for a split.
- `random_state`: reproducibility.
- `n_jobs`: number of CPU cores to use; `-1` uses all available cores.

Random Forest does not require feature scaling.

In [ ]:
forest_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

forest_model.fit(X_train_tree, y_train_tree)

forest_train_pred = forest_model.predict(X_train_tree)
forest_test_pred = forest_model.predict(X_test_tree)

results.append(
    evaluate_regression("Random Forest", y_test_tree, forest_test_pred)
)

print("Training R2:", r2_score(y_train_tree, forest_train_pred))
print("Testing R2:", r2_score(y_test_tree, forest_test_pred))

### Experiment: Random Forest hyperparameters

Compare different numbers of trees, maximum depths, and minimum leaf sizes.

In [ ]:
forest_experiment_results = []

for n_trees in [50, 100, 200]:
    for depth in [5, 10, None]:
        model = RandomForestRegressor(
            n_estimators=n_trees,
            max_depth=depth,
            min_samples_leaf=1,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        model.fit(X_train_tree, y_train_tree)

        train_predictions = model.predict(X_train_tree)
        test_predictions = model.predict(X_test_tree)

        forest_experiment_results.append({
            "n_estimators": n_trees,
            "max_depth": depth,
            "Train R2": r2_score(y_train_tree, train_predictions),
            **evaluate_regression(
                f"Forest trees={n_trees}, depth={depth}",
                y_test_tree,
                test_predictions
            )
        })

forest_experiment_df = pd.DataFrame(forest_experiment_results)
display(forest_experiment_df.sort_values("RMSE"))

## 13. Final model comparison

The following table compares the six main model configurations used above. The hyperparameter experiments are shown separately in their respective sections.

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="RMSE", y="Model")
plt.title("Model Comparison – RMSE (lower is better)")
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="R2", y="Model")
plt.title("Model Comparison – R² (higher is better)")
plt.show()

### Final discussion

Write short answers to the following:

1. Which model performed best?
2. Which model had the largest prediction errors?
3. Did Polynomial Regression outperform ordinary Linear Regression?
4. Did Ridge and Lasso improve the results?
5. Did Random Forest outperform a single Decision Tree?
6. Which model would you choose for this problem, and why?

**Evaluation reminder:** We used the test set for simple comparisons to keep this beginner exercise approachable. In a more rigorous workflow, use a validation set or cross-validation to tune hyperparameters, and evaluate the final selected model on the test set only once.

# 14. Optional Challenge – Pipeline and ColumnTransformer

In the traditional approach, we manually imputed, scaled, and encoded the features. Scikit-learn can combine these steps into a reusable workflow.

- `Pipeline` chains preprocessing and model training steps.
- `ColumnTransformer` applies different transformations to different columns.

In this section, we will use the original `X_train` and `X_test` DataFrames. Do not pass the manually transformed matrices to this pipeline.

## Step 1 – Import the necessary tools

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

## Step 2 – Identify feature groups

We already identified these columns earlier. Here we define them again for clarity.

In [ ]:
numerical_features = ["age", "bmi", "children"]
categorical_features = ["sex", "smoker", "region"]

## Step 3 – Create preprocessing pipelines

Numerical features will be imputed with their median and standardized. Categorical features will be imputed with their most frequent value and one-hot encoded.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

## Step 4 – Combine transformations with ColumnTransformer

The `transformers` list contains:
1. A name for the transformation.
2. The preprocessing pipeline.
3. The columns to which it should be applied.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

## Step 5 – Build a complete model Pipeline

The first step preprocesses the data. The second step trains the regression model.

Start with Ridge Regression.

In [ ]:
ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=1.0))
])

ridge_pipeline.fit(X_train, y_train)

ridge_pipeline_predictions = ridge_pipeline.predict(X_test)

display(pd.DataFrame([
    evaluate_regression(
        "Ridge Pipeline",
        y_test,
        ridge_pipeline_predictions
    )
]))

## Step 6 – Replace the model

One advantage of this approach is that you can reuse the same preprocessing workflow with a different estimator.

Try Random Forest.

In [ ]:
forest_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

forest_pipeline.fit(X_train, y_train)

forest_pipeline_predictions = forest_pipeline.predict(X_test)

display(pd.DataFrame([
    evaluate_regression(
        "Random Forest Pipeline",
        y_test,
        forest_pipeline_predictions
    )
]))

## Final optional challenge

1. Compare the traditional Ridge implementation with Ridge Pipeline.
2. Compare the traditional Random Forest implementation with Random Forest Pipeline.
3. Are their results consistent?
4. Try changing the estimator inside the Pipeline to Lasso or Decision Tree without changing the preprocessing workflow.

**Remember:** Tree-based models do not need feature scaling. You can create a separate tree preprocessing workflow without the scaler as an additional improvement.

### Why use Pipeline?

A Pipeline helps keep preprocessing and model training together. When you call `fit()` on the Pipeline, each preprocessing step is fitted using the training data, followed by the estimator. When you call `predict()`, the same fitted transformations are applied to new data automatically.

This makes the workflow easier to reuse and helps prevent preprocessing leakage.

# 15. Submission checklist

- [ ] Dataset loaded and inspected.
- [ ] EDA visualizations and observations included.
- [ ] Training and testing split created.
- [ ] Numerical and categorical preprocessing implemented.
- [ ] Linear Regression trained and evaluated.
- [ ] Polynomial Regression trained and evaluated.
- [ ] Ridge and Lasso regularization experiments completed.
- [ ] Decision Tree hyperparameter experiments completed.
- [ ] Random Forest hyperparameter experiments completed.
- [ ] Model comparison table included.
- [ ] Final discussion answered.
- [ ] Optional Pipeline section completed (bonus).

**The goal is not just to obtain the highest score. Understand what each model learns, how its parameters affect its behavior, and why one approach may be more suitable than another.**